<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/cse465/blob/main/RAG/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -U langchain


# Load website

In [4]:
!pip install -U langchain-community beautifulsoup4


In [5]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://notes.kodekloud.com/docs/LangChain/Performing-Retrieval/RAG-with-PDFs/page"

loader = WebBaseLoader(url)
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:1000])
print(docs[0].metadata)



/tmp/ipykernel_6183/758241715.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


1
RAG with PDFs - KodeKloudDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentKodeKloud home pageBetaSearch...⌘KSearch...NavigationPerforming RetrievalRAG with PDFsHomeCoursesLangChainContact UsCommunityBlogIntroductionCourse IntroductionConfiguring the Development EnvironmentOverview of LangChainWhat is LangChainWhy LangChainKey Use CasesKey Components of LangChainBuilding Blocks of LangChainModel IOMemoryRetrievalChainsToolsAgentsInteracting with LLMsSetting up EnvironmentTypes of Language ModelsGetting Started with LLMsMessages in ChatModelMessages in ChatModel DemoPrompt TemplatesPrompt Templates DemoFew shot Prompt TemplatesParsing Model OutputParsing Model Output Demo 1Parsing Model Output Demo 2Parsing Model Output Demo 3Tips, Tricks, and ResourcesKey LibrariesKey Libraries DemoTips and TricksDebugging LangChain ApplicationsUsing Verbose FlagCallbacksResourcesIntroducti

# Split the website into chunks

In [6]:
!pip install -U langchain-text-splitters

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(docs)

print(f"Documents: {len(docs)}")
print(f"Chunks: {len(chunks)}")
print(chunks[0].page_content)


Documents: 1
Chunks: 14
RAG with PDFs - KodeKloudDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentKodeKloud home pageBetaSearch...⌘KSearch...NavigationPerforming RetrievalRAG with PDFsHomeCoursesLangChainContact UsCommunityBlogIntroductionCourse IntroductionConfiguring the Development EnvironmentOverview of LangChainWhat is LangChainWhy LangChainKey Use CasesKey Components of LangChainBuilding Blocks of LangChainModel IOMemoryRetrievalChainsToolsAgentsInteracting with LLMsSetting up EnvironmentTypes of Language ModelsGetting Started with LLMsMessages in ChatModelMessages in ChatModel DemoPrompt TemplatesPrompt Templates DemoFew shot Prompt TemplatesParsing Model OutputParsing Model Output Demo 1Parsing Model Output Demo 2Parsing Model Output Demo 3Tips, Tricks, and ResourcesKey LibrariesKey Libraries DemoTips and TricksDebugging LangChain ApplicationsUsing Verbose FlagCallba

# Turn text into vectors

In [8]:
!pip install -U langchain-huggingface


In [9]:
!pip install -U sentence-transformers


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
texts = [chunk.page_content for chunk in chunks]

vectors = embeddings.embed_documents(texts)

print(len(vectors[0]))
print(vectors[0])


384
[-0.02651744708418846, -0.04277633875608444, -0.002789234509691596, -0.04931862652301788, 0.008542112074792385, 0.01612781174480915, -0.02500721998512745, 0.044853851199150085, 0.00232189754024148, 0.005351859610527754, 0.01787862554192543, -0.04042641818523407, 0.016904227435588837, 0.0012647772673517466, 0.00623677670955658, -0.025543924421072006, -0.003997824154794216, 0.03373036906123161, 0.013253318145871162, -0.009730824269354343, 0.027660757303237915, 0.01884401962161064, 0.07429496198892593, -0.0038884838577359915, 0.02760537900030613, -0.05267442762851715, -0.02373455837368965, -0.058864302933216095, 0.013575008139014244, -0.06563223898410797, 0.04611324146389961, 0.09226436913013458, -0.008577740751206875, 0.018741730600595474, 0.027365446090698242, 0.04568186029791832, 0.03597051650285721, -0.08147984743118286, 0.053830575197935104, -0.04373633489012718, -0.09097941964864731, -0.05676482245326042, -0.017894277349114418, -0.04633399844169617, 0.014642763882875443, -0.0266

# Store the vectors in FAISS


In [12]:
!pip install -U faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.0 MB/s eta 0:00:00


In [13]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)


In [14]:
query = "How to load PDF?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)



--- Result 1 ---
​High-level workflow

Load the PDF and split it into pages.
Chunk pages into smaller passages to control context length.
Embed chunks with an embeddings model.
Index embeddings into a vector store (Chroma).
Create a retriever from the vector store to fetch relevant passages at query time.
Format retrieved passages into a single context string.
Compose an LCEL chain: retriever -> formatter -> prompt -> LLM -> output parser.
Ask questions; the chain returns answers grounded in the document.

​Step-by-step implementation
​1) Load the PDF and split into page documents
loader = PyPDFLoader("data/handbook.pdf")
pages = loader.load_and_split()

This produces a list of page-level Document objects that preserve page content and metadata.
​2) Create chunks from the pages
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

--- Result 2 ---
​Quick reference: Steps & commands
StepPurposeExample command/sni

# Generate answer from LLM

In [15]:
!pip install -U transformers


In [16]:
from transformers import pipeline

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_id,
    device=-1
)

context = "\n\n".join(
    doc.page_content
    for doc in results
)
print(context)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

​High-level workflow

Load the PDF and split it into pages.
Chunk pages into smaller passages to control context length.
Embed chunks with an embeddings model.
Index embeddings into a vector store (Chroma).
Create a retriever from the vector store to fetch relevant passages at query time.
Format retrieved passages into a single context string.
Compose an LCEL chain: retriever -> formatter -> prompt -> LLM -> output parser.
Ask questions; the chain returns answers grounded in the document.

​Step-by-step implementation
​1) Load the PDF and split into page documents
loader = PyPDFLoader("data/handbook.pdf")
pages = loader.load_and_split()

This produces a list of page-level Document objects that preserve page content and metadata.
​2) Create chunks from the pages
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

​Quick reference: Steps & commands
StepPurposeExample command/snippetLoad PDFCreate page-level docum

In [17]:
prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

output = generator(
    prompt,
    max_new_tokens=200,
    do_sample=False
)

answer = output[0]["generated_text"][len(prompt):]

print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


To load a PDF file, you can use the `PyPDFLoader` class provided by LangChain. Here's how you can do it:

```python
from langchain.docx import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.retrievers import VectorSearchRetriever
from langchain.prompts import PromptTemplate
from langchain.chains import LabeledChain

# Step 1: Load the PDF and split it into page documents
pdf_loader = PyPDFLoader("data/handbook.pdf")
pages = pdf_loader.load_and_split()

# Step 2: Create chunks from the pages
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

# Summary
```

In this code snippet, we first import necessary modules from Lang
